In [2]:
from one.api import ONE
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from brainbox.ephys_plots import plot_brain_regions
from iblatlas.atlas import AllenAtlas
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainwidemap.bwm_loading import merge_probes
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.task.trials import find_trial_ids
from brainbox.io.one import SessionLoader
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox.singlecell import bin_spikes2D
import numpy as np
from iblatlas.atlas import BrainRegions
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import itertools
import pickle as pkl
from tqdm import tqdm
from pathlib import Path
import warnings
from sklearn.ensemble import RandomForestClassifier
from ibl_info.prepare_data_pid import (
    cleaned_regions_flags,
    get_new_cinc_intervals,
    get_window,
    prepare_ephys_data,
    get_congruent_incongruent_intervals,
)
from ibl_info.utils import (
    alternate_discretize,
    compute_mutual_information,
    compute_pid,
    compute_trivariate_mi,
    FIRING_RATE,
    discretize,
    equipopulated_binning,
)
import os
import concurrent.futures
import functools
import random
from glob import glob
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# the plan is to use cells that pass the significance tests
# these results are crazy prohibitive though
# the pid results should come thick and fast

In [3]:
df = pd.read_csv("../data/processed/significance_results_004d8fd5-41e7-4f1b-a45b-0d4ad76fe446.csv")

In [5]:
session_id = "004d8fd5-41e7-4f1b-a45b-0d4ad76fe446"
one = ONE()

In [6]:
pids, probes = one.eid2pid(session_id)
if isinstance(probes, list) and len(probes) > 1:
    to_merge = [load_good_units(one, pid=pid, qc=1) for pid in pids]
    spikes, clusters = merge_probes(
        [spikes for spikes, _ in to_merge], [clusters for _, clusters in to_merge]
    )
else:
    spikes, clusters = load_good_units(one, pid=pids[0], qc=1)

trials, mask = load_trials_and_mask(one, session_id, exclude_nochoice=True, exclude_unbiased=True)

In [7]:
epoch = "stim"
important_regions = [
    "VISp",
    "MOs",
    "SSp-ul",
    "ACAd",
    "PL",
    "CP",
    "VPM",
    "MG",
    "LGd",
    "ZI",
    "SNr",
    "MRN",
    "SCm",
    "PAG",
    "APN",
    "RN",
    "PPN",
    "PRNc",
    "PRNr",
    "GRN",
    "IRN",
    "PGRN",
    "CUL4 5",
    "SIM",
    "IP",
]

In [8]:
intervals, target_variable, congruent_flags, incongruent_flags = get_new_cinc_intervals(
    trials, epoch
)

In [9]:
for region in important_regions:
    binned_spikes, actual_regions, n_units, cluster_uuids_list = prepare_ephys_data(
        spikes, clusters, intervals, [region], minimum_units=5
    )

    if len(binned_spikes) == 0:
        # return empty arrays
        continue
    else:
        break

Region found IP, 14


In [10]:
nice_neurons = df[df["p_value"] < 0.05]["QC_cluster_id"].values

In [22]:
nxs = len(list(set(cluster_uuids_list[0]).intersection(set(nice_neurons))))

In [23]:
nxs

2